In [1]:
!pip install tensorflow numpy pandas matplotlib seaborn


In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import tensorflow as tf
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense
from collections import deque
import random


In [5]:

data = pd.read_csv("StudentsPerformance.csv")


data_encoded = pd.get_dummies(data, drop_first=True)


for col in ["math score", "reading score", "writing score"]:
    if col in data_encoded.columns:
        data_encoded[col] = data_encoded[col] / 100.0

print("Shape after preprocessing:", data_encoded.shape)
data_encoded.head()


Shape after preprocessing: (1000, 15)


,math score,reading score,writing score,gender_male,race/ethnicity_group B,race/ethnicity_group C,race/ethnicity_group D,race/ethnicity_group E,parental level of education_bachelor's degree,parental level of education_high school,parental level of education_master's degree,parental level of education_some college,parental level of education_some high school,lunch_standard,test preparation course_none
0,0.72,0.72,0.74,False,True,False,False,False,True,False,False,False,False,True,True
1,0.69,0.90,0.88,False,False,True,False,False,False,False,False,True,False,True,False
2,0.90,0.95,0.93,False,True,False,False,False,False,False,True,False,False,True,True
3,0.47,0.57,0.44,True,False,False,False,False,False,False,False,False,False,False,True
4,0.76,0.78,0.75,True,False,True,False,False,False,False,False,True,False,True,True


In [7]:
class TutorEnv:
    def __init__(self, data):
        self.data = data.values.astype(np.float32)
        self.num_students = len(data)
        self.num_features = data.shape[1]

        self.actions = ["practice", "revision", "new_topic"]
        self.action_space = len(self.actions)

        self.reset()

    def reset(self):
        self.current_index = np.random.randint(0, self.num_students)
        self.state = self.data[self.current_index].copy()
        return self.state

    def step(self, action):
        done = False

        # Simulated effect of each action
        if action == 0:  # practice
            change = np.random.uniform(0, 0.05)
        elif action == 1:  # revision
            change = np.random.uniform(0, 0.03)
        else:  # new topic
            change = np.random.uniform(-0.02, 0.07)

        reward = change
        self.state[:3] = np.clip(self.state[:3] + reward, 0, 1)

        if np.random.rand() < 0.1:  # end episode randomly
            done = True

        return self.state.copy(), reward, done


In [9]:
class DQNAgent:
    def __init__(self, state_size, action_size):
        self.state_size = state_size
        self.action_size = action_size
        self.memory = deque(maxlen=2000)

        self.gamma = 0.95
        self.epsilon = 1.0
        self.epsilon_min = 0.01
        self.epsilon_decay = 0.995
        self.learning_rate = 0.001

        self.model = self._build_model()
        self.target_model = self._build_model()
        self.update_target_model()

    def _build_model(self):
        model = Sequential([
            Dense(64, input_dim=self.state_size, activation="relu"),
            Dense(32, activation="relu"),
            Dense(self.action_size, activation="linear")
        ])
        model.compile(optimizer=tf.keras.optimizers.Adam(self.learning_rate), loss="mse")
        return model

    def update_target_model(self):
        self.target_model.set_weights(self.model.get_weights())

    def remember(self, state, action, reward, next_state, done):
        self.memory.append((state.copy(), action, reward, next_state.copy(), done))

    def act(self, state):
        if np.random.rand() <= self.epsilon:
            return np.random.randint(self.action_size)
        q_values = self.model.predict(state[np.newaxis, :], verbose=0)
        return np.argmax(q_values[0])

    def replay(self, batch_size=32):
        if len(self.memory) < batch_size:
            return

        minibatch = random.sample(self.memory, batch_size)
        states = np.array([m[0] for m in minibatch], dtype=np.float32)
        next_states = np.array([m[3] for m in minibatch], dtype=np.float32)

        target_q = self.model.predict(states, verbose=0)
        next_q = self.target_model.predict(next_states, verbose=0)

        for i, (_, action, reward, _, done) in enumerate(minibatch):
            target_q[i][action] = reward if done else reward + self.gamma * np.max(next_q[i])

        self.model.fit(states, target_q, epochs=1, verbose=0)

        if self.epsilon > self.epsilon_min:
            self.epsilon *= self.epsilon_decay


In [11]:
env = TutorEnv(data_encoded)
state_size = env.num_features
action_size = env.action_space

agent = DQNAgent(state_size, action_size)


C:\Users\SHYAM\anaconda3\Lib\site-packages\keras\src\layers\core\dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [ ]:
episodes = 50
batch_size = 32
scores = []

for e in range(episodes):
    state = env.reset()
    total_reward = 0
    for time in range(50):
        action = agent.act(state)
        next_state, reward, done = env.step(action)

        agent.remember(state, action, reward, next_state, done)
        state = next_state
        total_reward += reward

        if done:
            break

    agent.replay(batch_size)
    agent.update_target_model()
    scores.append(total_reward)

    print(f"Episode {e+1}/{episodes}, Reward: {total_reward:.2f}, Epsilon: {agent.epsilon:.2f}")


Episode 1/50, Reward: 0.04, Epsilon: 1.00
Episode 2/50, Reward: 0.06, Epsilon: 1.00
Episode 3/50, Reward: 0.14, Epsilon: 1.00
Episode 4/50, Reward: 0.12, Epsilon: 1.00
Episode 5/50, Reward: 0.15, Epsilon: 1.00
Episode 6/50, Reward: 0.14, Epsilon: 1.00
Episode 7/50, Reward: 0.08, Epsilon: 0.99
Episode 8/50, Reward: 0.38, Epsilon: 0.99
Episode 9/50, Reward: 0.07, Epsilon: 0.99
Episode 10/50, Reward: 0.08, Epsilon: 0.98
Episode 11/50, Reward: 0.27, Epsilon: 0.98
Episode 12/50, Reward: 0.24, Epsilon: 0.97
Episode 13/50, Reward: 0.18, Epsilon: 0.97
Episode 14/50, Reward: 0.06, Epsilon: 0.96
Episode 15/50, Reward: 0.32, Epsilon: 0.96
Episode 16/50, Reward: 0.00, Epsilon: 0.95
Episode 17/50, Reward: 0.25, Epsilon: 0.95
Episode 18/50, Reward: 0.17, Epsilon: 0.94
Episode 19/50, Reward: 0.14, Epsilon: 0.94
Episode 20/50, Reward: 0.70, Epsilon: 0.93
Episode 21/50, Reward: 0.83, Epsilon: 0.93
Episode 22/50, Reward: 0.04, Epsilon: 0.92
Episode 23/50, Reward: 0.06, Epsilon: 0.92
Episode 24/50, Rewar

In [ ]:
plt.plot(scores)
plt.xlabel("Episode")
plt.ylabel("Total Reward")
plt.title("Training Progress of Adaptive Tutor")
plt.show()


In [ ]:
state = env.reset()
total_reward = 0
recommendations = []

for _ in range(20):
    action = agent.act(state)
    next_state, reward, done = env.step(action)
    recommendations.append(env.actions[action])
    total_reward += reward
    state = next_state
    if done:
        state = env.reset()

print("Test Total Reward:", total_reward)
print("Recommended Study Paths:", recommendations)


In [ ]:
agent.model.save("adaptive_tutor_dqn.h5")
print("Model saved as adaptive_tutor_dqn.h5")
